In [2]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

In [ ]:
pip install zarr

In [ ]:
# 0. Preparation
###
# Importing libraries
import zarr
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

# Switch to GPU
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tue Aug  6 15:43:35 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   41C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                            

In [ ]:
# Reading data file from GoogleDrive
# Import zarr
zarr_data = zarr.open("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/data_1.1.zarr", mode='r')
zarr_target = zarr.open("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/target_1.1.zarr", mode='r')

# Work with zarr format directly
df = zarr_data
df_target = zarr_target

# Convert to pandas DataFrame - doesn't work, crashes due to RAM limit
###
#df = pd.DataFrame(zarr_data)
#df_target = pd.DataFrame(zarr_target)
#display(df.head())
#display(df_target.head())

# Define data name
df_name = "Baseline 1.1"

# Define random subsample for computation efficiency
#df = df.sample(500)
#df_target = df_target.sample(500)


In [ ]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Full size means we have now 90000 columns


<class 'zarr.core.Array'>
(21107, 89401)


In [ ]:
# Check missing values
#print(df.info())

#print("Missing vars in columns:\n", df.isna().sum())
#print("Number of total missing vars:", df.isna().sum().sum())
#print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# No missing vars. We can continue the ML modelling.

In [ ]:
# 1. Data preprocessing
###

# Create categorical variable from Case
#df_target = df_target.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
#df.Case.astype(int)

# Check construction
#print(df.Case.value_counts())

# Split data into target and features
#target = df.Case

# Features data: Drop Names and target
#data = df.drop(["Name", "Case"], axis = 1)
#data.head()
#data.shape




In [ ]:
# Moving Modelisation to GPU
###
import cupy as xp

# 1. Choose GPU Runtime

# 2. Split data into target and features
data = zarr_data
target = zarr_target

# 3. move data to GPU
data = xp.asarray(data)
target = xp.asarray(target)




In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)



In [ ]:
# PCA: Recuding dimensions
###

# We define n = 0.9 (so < 1) to retain 90 % of explained variance
pca = PCA(n_components = 0.9)

# Now apply the PCA on the training and test set.
# Attention: Make sure that the transformation to both test and train set is on the same N dimensions
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

# Show number of principal components retained
print("Number of components retained:", pca.n_components_)



KeyboardInterrupt: 

In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
#cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
#cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
#cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
#cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
#display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...